# 🔧 02 — Preprocessing Data

**Proyek:** Segmentasi UMKM Kota Bandung menggunakan K-Means Clustering

**Tujuan Notebook Ini:**
- Membersihkan data mentah per anggota (Indra, Dwi, Rajif)
- Filter wilayah Kota Bandung
- Menggabungkan dan deduplikasi data lintas anggota
- Analisis sentimen NLP pada ulasan
- Menyimpan data bersih ke `data/processed/`

> **Catatan:** Notebook ini HANYA berisi proses cleaning dan transformasi data. Visualisasi ada di `01_EDA_Data_Understanding.ipynb`, modeling di `03_Modeling_KMeans.ipynb`.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import sys
import warnings
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# Tambahkan path ke folder src/ untuk import modul
SRC_PATH = os.path.join('..', 'src')
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

RAW_PATH = os.path.join('..', 'data', 'raw')
PROCESSED_PATH = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

print('Library berhasil dimuat.')
print(f'Path data raw    : {os.path.abspath(RAW_PATH)}')
print(f'Path data output : {os.path.abspath(PROCESSED_PATH)}')

## 2. Fungsi Bantu Preprocessing

In [ ]:
def load_csv_safe(filename):
    """Load CSV dari folder data/raw/ dengan auto-detect delimiter dan abaikan utf-BOM."""
    path = os.path.join(RAW_PATH, filename)
    if not os.path.exists(path):
        print(f'  [SKIP] File tidak ditemukan: {filename}')
        return None
    try:
        df = pd.read_csv(path, sep=None, engine='python', encoding='utf-8-sig')
        df.columns = [c.replace('"', '').strip() for c in df.columns]
        return df
    except Exception as e:
        print(f'  [ERROR] Gagal memuat {filename}: {e}')
        return None


def load_member_raw_data(member_name):
    """Memuat pasangan file info tempat + ulasan per anggota."""
    info_list, review_list = [], []
    
    if member_name.lower() == 'indra':
        runs = [
            ('data_umkm_apify_scraper_no_text.csv', 'data_umkm_apify_scraper_ada_text_nya.csv'),
            ('data_umkm_apify_scraper_2_no_text.csv', 'data_umkm_apify_scraper_2_ada_text.csv')
        ]
    elif member_name.lower() == 'dwi':
        runs = [
            ('dataset_crawler-google-places_2026-06-18_15-33-25-280.csv', 'dataset_crawler-google-places_2026-06-18_15-33-25-280 (1).csv'),
            ('dataset_crawler-google-places_2026-06-19_02-47-57-745 (1).csv', 'dataset_crawler-google-places_2026-06-19_02-47-57-745 (2).csv'),
            ('dataset_crawler-google-places_2026-07-09_02-44-03-739.csv', 'dataset_crawler-google-places_2026-07-09_02-44-03-739 (1).csv')
        ]
    else:  # Rajif
        runs = [
            ('data overview1.csv', 'data review1.csv'),
            ('data overview2.csv', 'data review2.csv'),
            ('data overview3.csv', 'data review3.csv'),
            ('data overview4.csv', 'data review4.csv')
        ]
    
    for info_file, review_file in runs:
        df_info = load_csv_safe(info_file)
        df_review = load_csv_safe(review_file)
        if df_info is not None:
            info_list.append(df_info)
        if df_review is not None:
            review_list.append(df_review)
    
    df_info_all = pd.concat(info_list, ignore_index=True) if info_list else pd.DataFrame()
    df_review_all = pd.concat(review_list, ignore_index=True) if review_list else pd.DataFrame()
    
    return df_info_all, df_review_all

In [ ]:
def clean_member_data(df_info, df_review, member_name):
    """Membersihkan data satu anggota: filter Bandung, merge review, deduplikasi."""
    if df_info.empty:
        return pd.DataFrame()
    
    df_info = df_info.copy()
    
    # Pastikan kolom wajib ada
    required_cols = ['title', 'totalScore', 'reviewsCount', 'street', 'city', 'categoryName']
    for col in required_cols:
        if col not in df_info.columns:
            df_info[col] = ''
    
    # Konversi tipe data
    df_info['totalScore'] = pd.to_numeric(df_info['totalScore'].astype(str).str.replace(',', '.'), errors='coerce')
    med_score = df_info['totalScore'].median()
    df_info['totalScore'] = df_info['totalScore'].fillna(med_score if pd.notna(med_score) else 4.0)
    df_info['reviewsCount'] = pd.to_numeric(df_info['reviewsCount'].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)
    
    # Filter wilayah Kota Bandung
    df_info['city_clean'] = df_info['city'].astype(str).str.lower()
    df_info['street_clean'] = df_info['street'].astype(str).str.lower()
    mask_bandung = (
        df_info['city_clean'].str.contains('bandung', na=False) |
        df_info['street_clean'].str.contains('bandung', na=False) |
        (df_info['city_clean'] == '')
    )
    df_clean = df_info[mask_bandung].copy()
    
    # Buat entity_key untuk deduplikasi
    df_clean['entity_key'] = (
        df_clean['title'].astype(str).str.lower().str.strip() + '_' +
        df_clean['street'].astype(str).str.lower().str.strip()
    )
    
    # Gabungkan teks review
    if not df_review.empty and 'text' in df_review.columns:
        df_review = df_review.copy()
        if 'title' in df_review.columns:
            df_review['title_clean'] = df_review['title'].astype(str).str.lower().str.strip()
            reviews_grouped = df_review.groupby('title_clean')['text'].apply(
                lambda x: ' ||| '.join(x.dropna().astype(str))
            ).reset_index()
            df_clean['title_clean'] = df_clean['title'].astype(str).str.lower().str.strip()
            df_clean = pd.merge(df_clean, reviews_grouped, on='title_clean', how='left')
        else:
            df_clean['text'] = ''
    else:
        df_clean['text'] = ''
    
    df_clean['text'] = df_clean['text'].fillna('')
    df_clean['sumber_anggota'] = member_name
    
    kolom_final = ['entity_key', 'title', 'totalScore', 'reviewsCount', 'street', 'city', 'categoryName', 'text', 'sumber_anggota']
    return df_clean[kolom_final].drop_duplicates(subset=['entity_key'])

## 3. Tahap 01 — Preprocessing per Anggota

Membersihkan data mentah dari masing-masing anggota:
- Filter wilayah Kota Bandung
- Normalisasi kolom numerik
- Handle missing value
- Gabungkan teks ulasan

In [ ]:
members = ['Indra', 'Dwi', 'Rajif']
cleaned_dfs = {}

for member in members:
    print(f'\n--- Memproses data {member} ---')
    df_info, df_rev = load_member_raw_data(member)
    print(f'  Info Tempat : {len(df_info):,} baris')
    print(f'  Ulasan      : {len(df_rev):,} baris')
    
    df_clean = clean_member_data(df_info, df_rev, member)
    cleaned_dfs[member] = df_clean
    
    # Simpan hasil per anggota
    out_file = f'01_Hasil_Preprocessing_{member}_Final.csv'
    df_clean.to_csv(os.path.join(PROCESSED_PATH, out_file), index=False, sep=';', encoding='utf-8-sig')
    
    print(f'  Hasil bersih: {len(df_clean):,} UMKM')
    print(f'  Rating rerata: {df_clean["totalScore"].mean():.2f}')
    print(f'  Tersimpan   : {out_file}')

## 4. Tahap 02 — Penggabungan & Deduplikasi Lintas Anggota

Menggabungkan data 3 anggota dan menghapus entitas duplikat berdasarkan `entity_key` (judul + jalan).

In [ ]:
# Gabungkan semua anggota
dfs = [df for df in cleaned_dfs.values() if not df.empty]
df_all = pd.concat(dfs, ignore_index=True)
print(f'Total baris sebelum deduplikasi: {len(df_all):,}')

# Deduplikasi berdasarkan entity_key
grouped = df_all.groupby('entity_key')
records = []

for key, group in tqdm(grouped, desc='Deduplikasi Entitas Lintas Anggota'):
    first_row = group.iloc[0]
    reviews_combined = ' ||| '.join(set(
        [str(t).strip() for t in group['text'] if t and str(t).strip() != '']
    ))
    records.append({
        'entity_key': key,
        'title': first_row['title'],
        'totalScore': group['totalScore'].mean(),
        'reviewsCount': group['reviewsCount'].max(),
        'street': first_row['street'],
        'city': first_row['city'],
        'categoryName': first_row['categoryName'],
        'text': reviews_combined
    })

df_merged = pd.DataFrame(records)
print(f'Total entitas unik setelah deduplikasi: {len(df_merged):,}')

# Simpan
df_merged.to_csv(
    os.path.join(PROCESSED_PATH, '02_Data_Final_Sebelum_NLP_V2.csv'),
    index=False, sep=';', encoding='utf-8-sig'
)
print('Tersimpan: 02_Data_Final_Sebelum_NLP_V2.csv')

# Preview
df_merged[['title', 'totalScore', 'reviewsCount', 'city']].head()

## 5. Tahap 03 — Analisis Sentimen NLP

Menghitung skor sentimen dari teks ulasan menggunakan metode lexicon-based (Bahasa Indonesia).

In [ ]:
# Lexicon sentimen Bahasa Indonesia
KATA_POSITIF = {
    'enak', 'lezat', 'mantap', 'bagus', 'ramah', 'bersih', 'cepat', 'rekomendasi',
    'puas', 'murah', 'nyaman', 'suka', 'terbaik', 'top', 'wajib', 'juara',
    'strategis', 'lengkap', 'luas', 'adem', 'estetik', 'favorit', 'ramai'
}

KATA_NEGATIF = {
    'jelek', 'buruk', 'kecewa', 'mahal', 'lama', 'kotor', 'lambat', 'parah',
    'kurang', 'pahit', 'asin', 'bau', 'sempit', 'bising', 'kapok', 'rugi',
    'rusak', 'sombong', 'antri', 'macet', 'tutup', 'apatis'
}


def analyze_sentiment(text):
    """Hitung skor sentimen berdasarkan lexicon kata positif & negatif."""
    if not text or pd.isna(text):
        return 0.5
    # Note: escape character ganda digunakan karena format JSON notebook
    words = re.findall(r'\\b\\w+\\b', str(text).lower())
    if not words:
        return 0.5
    pos_count = sum(1 for w in words if w in KATA_POSITIF)
    neg_count = sum(1 for w in words if w in KATA_NEGATIF)
    total = pos_count + neg_count
    if total == 0:
        return 0.5
    score = (pos_count - neg_count) / total
    return round(0.5 + (score * 0.5), 4)


print('Menjalankan analisis sentimen NLP...')
scores = []
for text in tqdm(df_merged['text'], desc='Analisis Sentimen NLP'):
    scores.append(analyze_sentiment(text))

df_merged['sentiment_score'] = scores

# Statistik sentimen
print(f'\nSkor Sentimen Rerata   : {df_merged["sentiment_score"].mean():.4f}')
print(f'Skor Sentimen Minimum  : {df_merged["sentiment_score"].min():.4f}')
print(f'Skor Sentimen Maksimum : {df_merged["sentiment_score"].max():.4f}')

# Simpan
df_merged.to_csv(
    os.path.join(PROCESSED_PATH, '03_Data_Modeling_Setelah_NLP.csv'),
    index=False, sep=';', encoding='utf-8-sig'
)
print('\nTersimpan: 03_Data_Modeling_Setelah_NLP.csv')

## 6. Ringkasan Preprocessing

| Tahap | Input | Output | Keterangan |
|-------|-------|--------|------------|
| 01 | 18 file CSV mentah | `01_Hasil_Preprocessing_*_Final.csv` | Cleaning per anggota, filter Bandung |
| 02 | 3 file per anggota | `02_Data_Final_Sebelum_NLP_V2.csv` | Merge & deduplikasi lintas anggota |
| 03 | Data merged | `03_Data_Modeling_Setelah_NLP.csv` | Analisis sentimen NLP |

**Langkah selanjutnya:** Modeling K-Means Clustering di notebook `03_Modeling_KMeans.ipynb`.